In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [ ]:
# ======================
# Configuration
# ======================
DATA_PATH = "dataset"  # Update this path
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

# ======================
# Load dataset with validation split
# ======================
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

train_ds = image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = image_dataset_from_directory(
    DATA_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# ======================
# Model architecture
# ======================
model = Sequential([
    Rescaling(1./255),
    Conv2D(32, 3, activation='relu'),
    MaxPooling2D(),
    Conv2D(64, 3, activation='relu'),
    MaxPooling2D(),
    Conv2D(128, 3, activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(train_ds.class_names), activation='softmax')
])

# ======================
# Model compilation
# ======================
optimizer = Adam(learning_rate=0.0003)
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ======================
# Training
# ======================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

# ======================
# Evaluation
# ======================
val_acc = history.history['val_accuracy'][-1]
print(f"\nFinal Validation Accuracy: {val_acc*100:.2f}%")

# Save model unconditionally
model.save('trained_model.h5')
print("Model saved as 'trained_model.h5'")

Found 124486 files belonging to 4 classes.
Using 99589 files for training.
Found 124486 files belonging to 4 classes.
Using 24897 files for validation.
Epoch 1/20
1557/1557 [==============================] - 219s 140ms/step - loss: 0.4772 - accuracy: 0.8067 - val_loss: 0.3299 - val_accuracy: 0.8654
Epoch 2/20
1557/1557 [==============================] - 220s 141ms/step - loss: 0.2784 - accuracy: 0.8892 - val_loss: 0.2676 - val_accuracy: 0.8942
Epoch 3/20
1557/1557 [==============================] - 223s 143ms/step - loss: 0.1749 - accuracy: 0.9312 - val_loss: 0.2703 - val_accuracy: 0.8992
Epoch 4/20
1557/1557 [==============================] - 225s 144ms/step - loss: 0.1121 - accuracy: 0.9571 - val_loss: 0.2883 - val_accuracy: 0.9069
Epoch 5/20
1557/1557 [==============================] - 228s 147ms/step - loss: 0.0806 - accuracy: 0.9696 - val_loss: 0.3129 - val_accuracy: 0.9130
Epoch 6/20
1557/1557 [==============================] - 230s 148ms/step - loss: 0.0595 - accuracy: 0.9780 - 

In [ ]:
from tensorflow.keras.models import load_model

# Load the previously saved model
model = load_model('trained_model.h5')

# Continue training from epoch 21 to 50 (i.e., 30 more epochs)
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,            # total desired epochs
    initial_epoch=20      # resume from epoch 21
)


Epoch 21/50
1557/1557 [==============================] - 233s 149ms/step - loss: 0.0148 - accuracy: 0.9954 - val_loss: 0.6090 - val_accuracy: 0.9200
Epoch 22/50
1557/1557 [==============================] - 238s 153ms/step - loss: 0.0115 - accuracy: 0.9965 - val_loss: 0.6169 - val_accuracy: 0.9200
Epoch 23/50
1557/1557 [==============================] - 233s 149ms/step - loss: 0.0132 - accuracy: 0.9959 - val_loss: 0.6312 - val_accuracy: 0.9201
Epoch 24/50
1557/1557 [==============================] - 236s 152ms/step - loss: 0.0131 - accuracy: 0.9960 - val_loss: 0.6914 - val_accuracy: 0.9143
Epoch 25/50
1557/1557 [==============================] - 233s 149ms/step - loss: 0.0115 - accuracy: 0.9964 - val_loss: 0.5980 - val_accuracy: 0.9217
Epoch 26/50
1557/1557 [==============================] - 239s 153ms/step - loss: 0.0137 - accuracy: 0.9960 - val_loss: 0.6084 - val_accuracy: 0.9237
Epoch 27/50
1557/1557 [==============================] - 234s 150ms/step - loss: 0.0119 - accuracy: 0.9966

In [ ]:
from tensorflow.keras.models import load_model

# Load model saved after 50 epochs
model = load_model('trained_model_v2.h5')

# Continue training from epoch 51 to 100 (i.e., 50 more epochs)
history3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,           # total desired epochs
    initial_epoch=50      # resume from epoch 51
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Get predictions and true labels
y_true = []
y_pred = []

for images, labels in val_ds:
    predictions = model.predict(images)
    predicted_labels = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Step 2: Create confusion matrix (manual with NumPy)
num_classes = len(val_ds.class_names) # Use class names from the dataset
confusion_mtx = np.zeros((num_classes, num_classes), dtype=int)

for true, pred in zip(y_true, y_pred):
    confusion_mtx[true][pred] += 1

print("Confusion Matrix:")
print(confusion_mtx)

# Step 3: Plot confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(confusion_mtx, interpolation='nearest', cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(num_classes)
plt.xticks(tick_marks, val_ds.class_names, rotation=45) # Use class names from the dataset
plt.yticks(tick_marks, val_ds.class_names) # Use class names from the dataset

# Annotate cells
thresh = confusion_mtx.max() / 2.
for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, str(confusion_mtx[i, j]),
                 ha="center", va="center",
                 color="white" if confusion_mtx[i, j] > thresh else "black")

plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()